# Predictive Maintenance Using Machine Learning

##Data Preparation

### Research Question:
Can operational measurements be used to predict industrial equipment failure?

### Dataset:
AI4I 2020 Predictive Maintenance Dataset — UCI Machine Learning Repository.

The dataset contains 10,000 synthetic observations of industrial equipment operating conditions.

### Objectives:
1. Select suitable predictor variables.
2. Prevent target leakage.
3. Split the dataset into training and testing sets.
4. Preserve class proportions using stratified sampling.
5. Prepare a preprocessing pipeline for subsequent model training.

The dataset is synthetic, so model performance will be evaluated as a benchmark rather than evidence of real-world industrial deployment.

In [3]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

df = pd.read_csv("ai4i2020.csv")

print("Dataset shape:", df.shape)
display(df.head())


Dataset shape: (10000, 14)


,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


In [4]:

#numerical sensor measurements
numerical_features = [
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]"
]

#categorical features
categorical_features = ["Type"]

#predictor variables
X = df[numerical_features + categorical_features]

#target variable
y = df["Machine failure"]

print("Predictor shape:", X.shape)
print("Target shape:", y.shape)

display(X.head())


Predictor shape: (10000, 6)
Target shape: (10000,)


,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Type
0,298.1,308.6,1551,42.8,0,M
1,298.2,308.7,1408,46.3,3,L
2,298.1,308.5,1498,49.4,5,L
3,298.2,308.6,1433,39.5,7,L
4,298.2,308.7,1408,40.0,9,L


In [5]:

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training observations:", len(X_train))
print("Testing observations:", len(X_test))


Training observations: 8000
Testing observations: 2000


In [6]:

print("Original failure rate:")
print(f"{y.mean() * 100:.2f}%")

print("\nTraining failure rate:")
print(f"{y_train.mean() * 100:.2f}%")

print("\nTesting failure rate:")
print(f"{y_test.mean() * 100:.2f}%")

print("\nTraining class counts:")
print(y_train.value_counts())

print("\nTesting class counts:")
print(y_test.value_counts())


Original failure rate:
3.39%

Training failure rate:
3.39%

Testing failure rate:
3.40%

Training class counts:
Machine failure
0    7729
1     271
Name: count, dtype: int64

Testing class counts:
Machine failure
0    1932
1      68
Name: count, dtype: int64


In [7]:

# Define preprocessing for numerical variables
numeric_transformer = StandardScaler()

# Define preprocessing for categorical variables
categorical_transformer = OneHotEncoder(
    handle_unknown="ignore"
)

# Combine preprocessing steps
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

print("Preprocessing pipeline created successfully.")


Preprocessing pipeline created successfully.


## Data Preparation Methodology

The predictive features consist of five numerical operating measurements and one categorical machine-type variable.

The failure-type indicators (TWF, HDF, PWF, OSF and RNF) were excluded to prevent direct target leakage. UDI and Product ID were excluded because they are identifiers rather than operational measurements.

The dataset was divided into 80% training and 20% testing observations using stratified random sampling. Stratification was used to preserve the low machine failure rate across both subsets.

Numerical features will be standardized using StandardScaler, while the machine-type category will be encoded using OneHotEncoder.

Pre-processing will be fitted exclusively on the training data through a scikit-learn Pipeline during model development.

The held out test set will remain unused during model selection and tuning to provide a less biased final evaluation.

This aproach supports benchmark evaluation, eventhough a random split of synthetic observations does not re-inforce real world predictive performance.